In [47]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Customer Orders Products Project") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.2


In [48]:
customers_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(r"D:\Downloads\aws_s3_raw_dataset_1000_orders\customers_raw.csv")

orders_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(r"D:\Downloads\aws_s3_raw_dataset_1000_orders\orders_raw.csv")

products_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(r"D:\Downloads\aws_s3_raw_dataset_1000_orders\products_raw.csv")

In [49]:
customers_df.printSchema()
orders_df.printSchema()
products_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: double (nullable = true)



In [50]:
customers_df.show(5)
orders_df.show(5)
products_df.show(5)

+-----------+-------------+---------+-----------+-----------+
|customer_id|customer_name|     city|      state|signup_date|
+-----------+-------------+---------+-----------+-----------+
|       1001| Aarav Sharma|Bangalore|  Karnataka| 2025-01-01|
|       1002|     Diya Das|  Chennai| Tamil Nadu| 2025-02-02|
|       1003|   Rohan Bose|   Mumbai|Maharashtra| 2025-03-03|
|       1004| Ananya Patel|Hyderabad|  Telangana| 2025-04-04|
|       1005|  Vikram Shah|    Delhi|      Delhi| 2025-05-05|
+-----------+-------------+---------+-----------+-----------+
only showing top 5 rows
+--------+----------+-----------+----------+--------+----------+------------+------------+---------+-----------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|     city|      state|
+--------+----------+-----------+----------+--------+----------+------------+------------+---------+-----------+
|   50001|2026-03-21|       1018|      2016|       3|   42000.0|    126000.0|  

In [51]:
print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Products:", products_df.count())

Customers: 200
Orders: 1012
Products: 20


In [52]:
customers_df = customers_df.dropDuplicates()

orders_df = orders_df.dropDuplicates()

products_df = products_df.dropDuplicates()

In [53]:
print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Products:", products_df.count())

Customers: 200
Orders: 1001
Products: 20


In [54]:
from pyspark.sql.functions import col, sum

print("Customers NULL values:")
customers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_df.columns
]).show()

print("Orders NULL values:")
orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in orders_df.columns
]).show()

print("Products NULL values:")
products_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in products_df.columns
]).show()

Customers NULL values:
+-----------+-------------+----+-----+-----------+
|customer_id|customer_name|city|state|signup_date|
+-----------+-------------+----+-----+-----------+
|          0|            0|   0|    0|          0|
+-----------+-------------+----+-----+-----------+

Orders NULL values:
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|       0|        10|         10|         6|       0|         0|          10|           0|   0|    0|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+

Products NULL values:
+----------+------------+--------+-----------+----------+
|product_id|product_name|category|subcategory|unit_price|
+----------+------------+--------+-----------+------

In [55]:
orders_df = orders_df.dropna(
    subset=[
        "order_date",
        "customer_id",
        "product_id",
        "sales_amount"
    ]
)

In [56]:
print("Orders NULL values:")
orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in orders_df.columns
]).show()


Orders NULL values:
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|       0|         0|          0|         0|       0|         0|           0|           0|   0|    0|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [57]:
print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Products:", products_df.count())

Customers: 200
Orders: 965
Products: 20


In [58]:
orders_df.filter(col("quantity") < 0).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+---------+----------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|     city|     state|
+--------+----------+-----------+----------+--------+----------+------------+------------+---------+----------+
|   50121|2026-01-25|       1124|      2016|      -2|   42000.0|    126000.0|     Pending|Hyderabad| Telangana|
|   50464|2026-06-03|       1001|      2002|      -2|    1200.0|      2400.0|     Pending|Bangalore| Karnataka|
|   50098|2026-05-19|       1047|      2007|      -1|    2200.0|      4400.0|     Shipped|  Chennai|Tamil Nadu|
|   50710|2026-02-25|       1127|      2009|      -1|    3200.0|     16000.0|     Shipped|Ahmedabad|   Gujarat|
|   50643|2026-06-24|       1193|      2006|      -3|   18000.0|     18000.0|     Shipped|    Surat|   Gujarat|
+--------+----------+-----------+----------+--------+----------+------------+------------+---------+----

In [59]:
orders_df.filter(col("unit_price") < 0).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [60]:
orders_df.filter(col("sales_amount") < 0).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [61]:
orders_df = orders_df.filter(col("quantity") > 0)

In [62]:
orders_df.filter(col("quantity") < 0).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [63]:
print("Orders after cleaning:", orders_df.count())

Orders after cleaning: 955


In [64]:
orders_df.filter(
    col("sales_amount") != col("quantity") * col("unit_price")
).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [65]:
orders_df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     Shipped|
|   completed|
|   Completed|
|    COMPLETE|
|   Cancelled|
|     Pending|
+------------+



In [66]:
from pyspark.sql.functions import when,lower,initcap

orders_df = orders_df.withColumn(
    "order_status",
    when(lower(col("order_status")).isin("complete", "completed"), "Completed")
    .otherwise(initcap(lower(col("order_status"))))
)

In [67]:
orders_df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     Shipped|
|   Completed|
|   Cancelled|
|     Pending|
+------------+



In [68]:
orders_df.select("city").distinct().show(truncate=False)

+------------+
|city        |
+------------+
|Bangalore   |
|Kochi       |
|Bhubaneswar |
|Chennai     |
|Lucknow     |
|Mumbai      |
|Ahmedabad   |
|Kolkata     |
|Bhubaneswar |
|Surat       |
|Kochi       |
|Pune        |
|Delhi       |
|Nagpur      |
|Bhopal      |
|Jaipur      |
|Bhopal      |
|Hyderabad   |
|Jaipur      |
+------------+



In [69]:
orders_df.select("state").distinct().show(truncate=False)

+---------------+
|state          |
+---------------+
|Karnataka      |
|Odisha         |
|Kerala         |
|Tamil Nadu     |
| Maharashtra   |
|Madhya Pradesh |
| Madhya Pradesh|
| Telangana     |
| Delhi         |
|Gujarat        |
|Delhi          |
|Rajasthan      |
|Maharashtra    |
|West Bengal    |
|Telangana      |
|Uttar Pradesh  |
+---------------+



In [70]:
from pyspark.sql.functions import trim

orders_df = orders_df.withColumn(
    "state",
    trim(col("state"))
)

In [71]:
orders_df.select("state").distinct().show(truncate=False)

+--------------+
|state         |
+--------------+
|Karnataka     |
|Odisha        |
|Kerala        |
|Tamil Nadu    |
|Madhya Pradesh|
|Gujarat       |
|Delhi         |
|Rajasthan     |
|Maharashtra   |
|West Bengal   |
|Telangana     |
|Uttar Pradesh |
+--------------+



In [72]:
orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left_anti"
).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+---------+-----------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|     city|      state|
+--------+----------+-----------+----------+--------+----------+------------+------------+---------+-----------+
|   50968|2026-06-03|       9026|      2003|       1|    3500.0|      3500.0|   Cancelled|Bangalore|  Karnataka|
|   50590|2026-04-07|       9006|      2012|       2|   28000.0|     56000.0|     Shipped|Ahmedabad|    Gujarat|
|   50036|2026-06-13|       9025|      2015|       1|   12000.0|     12000.0|     Shipped|   Mumbai|Maharashtra|
|   50515|2026-03-12|       9012|      2019|       3|    9500.0|     28500.0|     Pending|    Kochi|     Kerala|
|   50639|2026-02-04|       9044|      2014|       3|    7500.0|     22500.0|     Pending|Bangalore|  Karnataka|
|   50528|2026-01-28|       9000|      2007|       3|    2200.0|      6600.0|   Cancelled|     P

In [73]:
orders_df.join(
    products_df,
    orders_df.product_id == products_df.product_id,
    "left_anti"
).show()

+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
|order_id|order_date|customer_id|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+
+--------+----------+-----------+----------+--------+----------+------------+------------+----+-----+



In [74]:
orders_df = orders_df.join(
    customers_df.select("customer_id"),
    on="customer_id",
    how="inner"
)

In [75]:
orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left_anti"
).show()

+-----------+--------+----------+----------+--------+----------+------------+------------+----+-----+
|customer_id|order_id|order_date|product_id|quantity|unit_price|sales_amount|order_status|city|state|
+-----------+--------+----------+----------+--------+----------+------------+------------+----+-----+
+-----------+--------+----------+----------+--------+----------+------------+------------+----+-----+



In [76]:
print("Orders after customer validation:", orders_df.count())

Orders after customer validation: 945


In [77]:
orders_df.join(
    products_df.select("product_id"),
    on="product_id",
    how="left_anti"
).show()

+----------+-----------+--------+----------+--------+----------+------------+------------+----+-----+
|product_id|customer_id|order_id|order_date|quantity|unit_price|sales_amount|order_status|city|state|
+----------+-----------+--------+----------+--------+----------+------------+------------+----+-----+
+----------+-----------+--------+----------+--------+----------+------------+------------+----+-----+



In [78]:
from pyspark.sql.functions import month

orders_df = orders_df.withColumn(
    "order_month",
    month(col("order_date"))
)

In [79]:
orders_df.select(
    "order_date",
    "order_month"
).show(10)

+----------+-----------+
|order_date|order_month|
+----------+-----------+
|2026-05-26|          5|
|2026-05-25|          5|
|2026-05-10|          5|
|2026-06-01|          6|
|2026-04-16|          4|
|2026-06-28|          6|
|2026-03-09|          3|
|2026-01-10|          1|
|2026-01-29|          1|
|2026-05-17|          5|
+----------+-----------+
only showing top 10 rows


In [80]:
from pyspark.sql.functions import date_format

orders_df = orders_df.withColumn(
    "order_month_name",
    date_format(col("order_date"), "MMMM")
)

In [81]:
orders_df.select(
    "order_date",
    "order_month",
    "order_month_name"
).show(10)

+----------+-----------+----------------+
|order_date|order_month|order_month_name|
+----------+-----------+----------------+
|2026-05-26|          5|             May|
|2026-05-25|          5|             May|
|2026-05-10|          5|             May|
|2026-06-01|          6|            June|
|2026-04-16|          4|           April|
|2026-06-28|          6|            June|
|2026-03-09|          3|           March|
|2026-01-10|          1|         January|
|2026-01-29|          1|         January|
|2026-05-17|          5|             May|
+----------+-----------+----------------+
only showing top 10 rows


In [82]:
orders = orders_df.alias("o")
customers = customers_df.alias("c")
products = products_df.alias("p")

In [83]:
final_df = (
    orders
    .join(
        customers,
        col("o.customer_id") == col("c.customer_id"),
        "inner"
    )
    .join(
        products,
        col("o.product_id") == col("p.product_id"),
        "inner"
    )
    .select(
        col("o.order_id"),
        col("o.order_date"),
        col("o.order_month"),
        col("o.order_month_name"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("c.city"),
        col("c.state"),
        col("c.signup_date"),
        col("o.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.subcategory"),
        col("o.quantity"),
        col("o.unit_price"),
        col("o.sales_amount"),
        col("o.order_status")
    )
)

In [84]:
final_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- order_month_name: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- order_status: string (nullable = true)



In [85]:
final_df.show(5, truncate=False)

+--------+----------+-----------+----------------+-----------+-------------+---------+-----------+-----------+----------+-------------------+-----------+-----------+--------+----------+------------+------------+
|order_id|order_date|order_month|order_month_name|customer_id|customer_name|city     |state      |signup_date|product_id|product_name       |category   |subcategory|quantity|unit_price|sales_amount|order_status|
+--------+----------+-----------+----------------+-----------+-------------+---------+-----------+-----------+----------+-------------------+-----------+-----------+--------+----------+------------+------------+
|50350   |2026-05-26|5          |May             |1196       |Aarav Sharma |Bangalore|Karnataka  |2025-04-21 |2020      |Monitor 27 inch    |Electronics|Monitors   |2       |22000.0   |44000.0     |Pending     |
|50713   |2026-05-25|5          |May             |1014       |Pooja Nair   |Nagpur   |Maharashtra|2025-02-14 |2004      |Monitor 24 inch    |Electronics

In [86]:
print("Final row count:", final_df.count())

Final row count: 945


In [87]:
from pyspark.sql.functions import col, sum

final_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in final_df.columns
]).show()

+--------+----------+-----------+----------------+-----------+-------------+----+-----+-----------+----------+------------+--------+-----------+--------+----------+------------+------------+
|order_id|order_date|order_month|order_month_name|customer_id|customer_name|city|state|signup_date|product_id|product_name|category|subcategory|quantity|unit_price|sales_amount|order_status|
+--------+----------+-----------+----------------+-----------+-------------+----+-----+-----------+----------+------------+--------+-----------+--------+----------+------------+------------+
|       0|         0|          0|               0|          0|            0|   0|    0|          0|         0|           0|       0|          0|       0|         0|           0|           0|
+--------+----------+-----------+----------------+-----------+-------------+----+-----+-----------+----------+------------+--------+-----------+--------+----------+------------+------------+



In [93]:
final_df.toPandas().to_csv(
    r"D:\Downloads\output\final_sales.csv",
    index=False
)

In [94]:
import os

print(os.path.exists(r"D:\Downloads\output\final_sales.csv"))

True
